# CBOW Data Preparation
Dataset preparation steps with CBOW in mind.

Used dataset: text8
https://huggingface.co/roshbeed/text8-dataset

## Config

In [1]:
WINDOW_SIZE = 1
MIN_FREQ = 5
SUBSAMPLE_T = 1e-5
TRAIN_SPLIT = 0.9

DATA_PATH = r"dataset\repo\text8_full.txt"

VOCAB_PATH = r"dataset\processed\vocab.json"
TRAIN_PATH = r"dataset\processed\train.csv"
TEST_PATH = r"dataset\processed\test.csv"


## Imports

In [2]:
import json, math, random, csv
from collections import Counter
from pathlib import Path

random.seed(42)


## 1. Vocabulary
Discard words below `MIN_FREQ`. Rare words are dropped entirely.

In [3]:
tokens_raw = Path(DATA_PATH).read_text(encoding="utf-8").strip().split()

freq = Counter(tokens_raw)
vocab_words = sorted([w for w, c in freq.items() if c >= MIN_FREQ], key=lambda w: freq[w], reverse=True)
word_to_idx = {w: i for i, w in enumerate(vocab_words)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
VOCAB_SIZE  = len(vocab_words)

print(f"Raw tokens   : {len(tokens_raw):,}")
print(f"Vocab size   : {VOCAB_SIZE:,}  (discarded {len(freq)-VOCAB_SIZE:,} rare words)")


Raw tokens   : 17,005,207
Vocab size   : 71,290  (discarded 182,564 rare words)


## 2. Subsampling
Probabilistically discard frequent words from the token stream using the Word2Vec formula: $P(\text{discard}) = 1 - \sqrt{t / f(w)}$. Applied to the corpus.

In [4]:
total = sum(freq[w] for w in vocab_words)
freq_ratio = {w: freq[w] / total for w in vocab_words}

def keep_prob(w): return min(1.0, math.sqrt(SUBSAMPLE_T / freq_ratio[w]))

tokens = [w for w in tokens_raw if w in word_to_idx and random.random() < keep_prob(w)]
print(f"Tokens after subsampling : {len(tokens):,}  ({len(tokens)/len(tokens_raw)*100:.1f}% retained)")


Tokens after subsampling : 4,667,401  (27.4% retained)


## 3. Save vocabulary

In [5]:
with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "vocab_size"  : VOCAB_SIZE,
        "word_to_idx" : word_to_idx,
        "idx_to_word" : {str(i): w for i, w in idx_to_word.items()},
    }, f, ensure_ascii=False)

print(f"Saved '{VOCAB_PATH}'  ({VOCAB_SIZE:,} entries)")


Saved 'dataset\processed\vocab.json'  (71,290 entries)


## 4. Sliding window
Treat the full subsampled corpus as one sequence. Each window yields one `(X, y)` pair where `X` is the `2 * WINDOW_SIZE` context indices and `y` is the centre index.

In [6]:
indices = [word_to_idx[w] for w in tokens]

X, y = [], []
for t in range(WINDOW_SIZE, len(indices) - WINDOW_SIZE):
    context = indices[t - WINDOW_SIZE : t] + indices[t + 1 : t + WINDOW_SIZE + 1]
    X.append(context)
    y.append(indices[t])

print(f"Total windows : {len(X):,}")
print(f"Context size  : {2*WINDOW_SIZE}")
print(f"Sample X[0]   : {X[0]}  -> {[idx_to_word[i] for i in X[0]]}")
print(f"Sample y[0]   : {y[0]}  -> {idx_to_word[y[0]]}")


Total windows : 4,667,399
Context size  : 2
Sample X[0]   : [5233, 155]  -> ['anarchism', 'against']
Sample y[0]   : 3080  -> originated


## 5. Shuffle & split

In [7]:
order = list(range(len(X)))
random.shuffle(order)

split = int(len(order) * TRAIN_SPLIT)
train_idx, test_idx = order[:split], order[split:]

X_train = [X[i] for i in train_idx];  y_train = [y[i] for i in train_idx]
X_test  = [X[i] for i in test_idx];   y_test  = [y[i] for i in test_idx]

print(f"Train : {len(X_train):,}   Test : {len(X_test):,}")


Train : 4,200,659   Test : 466,740


## 6. Dump to CSV
Context indices stored as space-separated values in a single `x` column.

In [9]:
def write_csv(path, X, y):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["x", "y"])
        for xi, yi in zip(X, y):
            writer.writerow([" ".join(map(str, xi)), yi])

write_csv(TRAIN_PATH, X_train, y_train)
write_csv(TEST_PATH,  X_test,  y_test)
print(f"Saved {TRAIN_PATH} and {TEST_PATH}")


Saved dataset\processed\train.csv and dataset\processed\test.csv
